# Discovery y Perfilado de Datos — Sistema Billing

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

In [2]:
user = os.environ.get("WAREHOUSE_DB_USER", "postgres")
password = os.environ.get("WAREHOUSE_DB_PASSWORD", "postgres")
db = os.environ.get("WAREHOUSE_DB_NAME", "warehouse")
host = os.environ.get("WAREHOUSE_DB_HOST", "warehouse-db")
port = os.environ.get("WAREHOUSE_DB_PORT", "5432")

In [3]:
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}")

### Tabla costumer y relación con Tabla student

In [4]:
#Cuántos customers tienen external_ref
query = """
SELECT
  COUNT(*) AS total_customers,
  COUNT(*) FILTER (WHERE external_ref IS NULL) AS sin_external_ref,
  COUNT(*) FILTER (WHERE external_ref IS NOT NULL) AS con_external_ref
FROM bronze.billing_customers;
"""
df = pd.read_sql(query, engine)
df

,total_customers,sin_external_ref,con_external_ref
0,10000,5000,5000


In [5]:
#external_ref hacen match real contra university.students
query = """
SELECT
  COUNT(*) AS con_external_ref,
  COUNT(s.student_id) AS con_match_en_university,
  COUNT(*) - COUNT(s.student_id) AS sin_match
FROM bronze.billing_customers c
LEFT JOIN bronze.university_students s
  ON s.student_id::text = c.external_ref
WHERE c.external_ref IS NOT NULL;
"""
df = pd.read_sql(query, engine)
df

,con_external_ref,con_match_en_university,sin_match
0,5000,5000,0


In [6]:
#external_ref único, o se repite entre varios customer_id
query = """
SELECT external_ref, COUNT(*) AS n_customers
FROM bronze.billing_customers
WHERE external_ref IS NOT NULL
GROUP BY external_ref
HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,external_ref,n_customers


In [7]:
#País del cliente vs. país del estudiante (sobre los que sí matchean)
query = """
SELECT
  COUNT(*) AS total_match,
  COUNT(*) FILTER (WHERE c.country = s.country) AS mismo_pais,
  COUNT(*) FILTER (WHERE c.country <> s.country) AS distinto_pais
FROM bronze.billing_customers c
JOIN bronze.university_students s
  ON s.student_id::text = c.external_ref;
"""
df = pd.read_sql(query, engine)
df

,total_match,mismo_pais,distinto_pais
0,5000,1072,3928


In [8]:
query = """
SELECT c.customer_id, c.external_ref, c.country AS pais_cliente, s.country AS pais_estudiante
FROM bronze.billing_customers c
JOIN bronze.university_students s
  ON s.student_id::text = c.external_ref
WHERE c.country <> s.country;
"""
df = pd.read_sql(query, engine)
df

,customer_id,external_ref,pais_cliente,pais_estudiante
0,CUS-0000001,STU-0000001,CL,US
1,CUS-0000002,STU-0000002,MX,CL
2,CUS-0000003,STU-0000003,CL,PE
3,CUS-0000004,STU-0000004,MX,CL
4,CUS-0000006,STU-0000006,CL,AR
...,...,...,...,...
3923,CUS-0004993,STU-0004993,PE,MX
3924,CUS-0004994,STU-0004994,MX,CL
3925,CUS-0004996,STU-0004996,PE,CO
3926,CUS-0004998,STU-0004998,ES,PE


### Tabla customers

In [9]:
#Perfil general
query = """
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT customer_id) AS ids,
  COUNT(*) FILTER (WHERE email IS NULL) AS email,
  COUNT(*) FILTER (WHERE country IS NULL) AS country,
  COUNT(*) FILTER (WHERE segment IS NULL) AS segment,
  COUNT(*) FILTER (WHERE created_at IS NULL) AS created_at
FROM bronze.billing_customers;
"""
df = pd.read_sql(query, engine)
df

,total,ids,email,country,segment,created_at
0,10000,10000,0,0,0,0


In [10]:
#Duplicados de email
query = """
SELECT email, COUNT(*) AS n
FROM bronze.billing_customers
GROUP BY email
HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,email,n


In [11]:
#Cardinalidad de segment
query = """
SELECT segment, COUNT(*) AS n
FROM bronze.billing_customers
GROUP BY segment
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,segment,n
0,retail,7019
1,smb,2200
2,enterprise,781


In [12]:
#Validar formato de created_at antes de castear
query = """
SELECT created_at, COUNT(*) AS n
FROM bronze.billing_customers
WHERE created_at IS NOT NULL AND created_at !~ '^\d{4}-\d{2}-\d{2}'
GROUP BY created_at;
"""
df = pd.read_sql(query, engine)
df

,created_at,n


In [13]:
#Rango de created_at (una vez validado el formato)
query = """
SELECT
  MIN(created_at::date) AS cliente_mas_antiguo,
  MAX(created_at::date) AS cliente_mas_reciente
FROM bronze.billing_customers;
"""
df = pd.read_sql(query, engine)
df

,cliente_mas_antiguo,cliente_mas_reciente
0,2018-01-01,2025-12-30


In [14]:
query = """
SELECT date_trunc('month', created_at::date) AS mes, COUNT(*) AS n_clientes
FROM bronze.billing_customers
GROUP BY 1
ORDER BY 1;
"""
df = pd.read_sql(query, engine)
df

,mes,n_clientes
0,2018-01-01 00:00:00+00:00,98
1,2018-02-01 00:00:00+00:00,96
2,2018-03-01 00:00:00+00:00,102
3,2018-04-01 00:00:00+00:00,82
4,2018-05-01 00:00:00+00:00,104
...,...,...
91,2025-08-01 00:00:00+00:00,104
92,2025-09-01 00:00:00+00:00,117
93,2025-10-01 00:00:00+00:00,106
94,2025-11-01 00:00:00+00:00,99


In [15]:
#Cardinalidad de country
query = """
SELECT country, COUNT(*) AS n
FROM bronze.billing_customers
GROUP BY country
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,country,n
0,CL,4053
1,PE,991
2,AR,989
3,MX,969
4,BR,817
5,ES,790
6,CO,775
7,US,616


### Tabla productos

In [16]:
#Perfil general
query = """
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT product_id) AS ids,
  COUNT(DISTINCT sku) AS distinct_skus,
  COUNT(*) FILTER (WHERE category IS NULL) AS category,
  COUNT(*) FILTER (WHERE monthly_price IS NULL) AS price,
  COUNT(*) FILTER (WHERE active IS NULL) AS active
FROM bronze.billing_products;
"""
df = pd.read_sql(query, engine)
df

,total,ids,distinct_skus,category,price,active
0,200,200,200,0,0,0


In [17]:

#Duplicados de sku
query = """
SELECT sku, COUNT(*) AS n
FROM bronze.billing_products
GROUP BY sku
HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,sku,n


In [18]:
#Cardinalidad de category
query = """
SELECT category, COUNT(*) AS n
FROM bronze.billing_products
GROUP BY category
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,category,n
0,basic,86
1,standard,58
2,premium,37
3,enterprise,19


In [19]:
#Distribución de active (True/False)
query = """
SELECT active::boolean AS activo, COUNT(*) AS n
FROM bronze.billing_products
GROUP BY 1;
"""
df = pd.read_sql(query, engine)
df

,activo,n
0,False,30
1,True,170


In [20]:
#Distribución de monthly_price
query = """
SELECT
  MIN(monthly_price::numeric) AS precio_min,
  MAX(monthly_price::numeric) AS precio_max,
  AVG(monthly_price::numeric) AS precio_promedio
FROM bronze.billing_products;
"""
df = pd.read_sql(query, engine)
df

,precio_min,precio_max,precio_promedio
0,5.0,491.47,46.2465


In [21]:
query = """
SELECT category,
       MIN(monthly_price::numeric) AS precio_min,
       MAX(monthly_price::numeric) AS precio_max,
       AVG(monthly_price::numeric) AS precio_promedio
FROM bronze.billing_products
GROUP BY category
ORDER BY category;
"""
df = pd.read_sql(query, engine)
df

,category,precio_min,precio_max,precio_promedio
0,basic,5.00,491.47,51.478140
1,enterprise,5.00,71.24,36.759474
2,premium,7.86,219.41,43.973514
3,standard,6.15,198.27,43.047069


### Tabla subscriptions

In [22]:
#Perfil general
query = """
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT subscription_id) AS ids,
  COUNT(*) FILTER (WHERE status IS NULL) AS status,
  COUNT(*) FILTER (WHERE start_date IS NULL) AS start_date,
  COUNT(*) FILTER (WHERE end_date IS NULL) AS end_date,
  COUNT(*) FILTER (WHERE customer_id IS NULL) AS customer_id,
  COUNT(*) FILTER (WHERE product_id IS NULL) AS product_id
FROM bronze.billing_subscriptions;
"""
df = pd.read_sql(query, engine)
df

,total,ids,status,start_date,end_date,customer_id,product_id
0,15000,15000,0,0,0,0,0


In [23]:
#Cardinalidad de status
query = """
SELECT status, COUNT(*) AS n
FROM bronze.billing_subscriptions
GROUP BY status
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,status,n
0,active,11272
1,cancelled,2242
2,paused,1486


In [24]:
#Validar formato de fechas antes de castear
query = """
SELECT start_date, COUNT(*) AS n
FROM bronze.billing_subscriptions
WHERE start_date IS NOT NULL AND start_date !~ '^\d{4}-\d{2}-\d{2}'
GROUP BY start_date;
"""
df = pd.read_sql(query, engine)
df

,start_date,n


In [25]:
query = """
SELECT end_date, COUNT(*) AS n
FROM bronze.billing_subscriptions
WHERE end_date IS NOT NULL AND end_date !~ '^\d{4}-\d{2}-\d{2}'
GROUP BY end_date;
"""
df = pd.read_sql(query, engine)
df

,end_date,n


In [26]:
#start_date debe ser menor que end_date
query = """
SELECT *
FROM bronze.billing_subscriptions
WHERE end_date IS NOT NULL
  AND start_date::date >= end_date::date;
"""
df = pd.read_sql(query, engine)
df

,subscription_id,status,start_date,end_date,customer_id,product_id
0,SUB-0000029,active,2024-12-03,2024-02-19,CUS-0005963,PRD-00004
1,SUB-0000044,paused,2024-07-16,2024-07-06,CUS-0009149,PRD-00096
2,SUB-0000056,active,2024-05-27,2024-05-04,CUS-0009361,PRD-00102
3,SUB-0000062,active,2025-05-20,2024-04-12,CUS-0004678,PRD-00165
4,SUB-0000070,active,2025-04-16,2024-07-09,CUS-0003571,PRD-00118
...,...,...,...,...,...,...
784,SUB-0014929,active,2025-06-10,2024-05-13,CUS-0006966,PRD-00115
785,SUB-0014944,active,2025-06-10,2024-02-19,CUS-0001092,PRD-00138
786,SUB-0014963,active,2024-12-22,2024-09-22,CUS-0000266,PRD-00040
787,SUB-0014966,active,2024-10-31,2024-02-19,CUS-0008734,PRD-00111


In [27]:
#Suscripciones "active" con end_date ya vencido (inconsistencia de negocio)
query = """
SELECT *
FROM bronze.billing_subscriptions
WHERE status = 'active'
  AND end_date IS NOT NULL
  AND end_date::date < CURRENT_DATE;
"""
df = pd.read_sql(query, engine)
df

,subscription_id,status,start_date,end_date,customer_id,product_id
0,SUB-0000001,active,2024-10-15,2026-05-14,CUS-0006082,PRD-00154
1,SUB-0000003,active,2021-02-27,2025-01-25,CUS-0004408,PRD-00064
2,SUB-0000004,active,2020-10-02,2025-05-23,CUS-0005220,PRD-00112
3,SUB-0000006,active,2024-01-02,2025-12-17,CUS-0004115,PRD-00053
4,SUB-0000007,active,2021-05-02,2024-08-05,CUS-0000547,PRD-00040
...,...,...,...,...,...,...
7149,SUB-0014989,active,2023-09-01,2025-10-28,CUS-0001591,PRD-00034
7150,SUB-0014992,active,2022-06-06,2024-09-20,CUS-0004542,PRD-00074
7151,SUB-0014994,active,2021-02-14,2024-02-07,CUS-0003001,PRD-00179
7152,SUB-0014997,active,2021-06-13,2025-02-13,CUS-0000065,PRD-00125


In [28]:
#Duplicados por clave de negocio (customer+product+start_date)
query = """
SELECT customer_id, product_id, start_date, COUNT(*) AS n
FROM bronze.billing_subscriptions
GROUP BY customer_id, product_id, start_date
HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,customer_id,product_id,start_date,n


In [29]:
#Integridad referencial
query = """
SELECT s.subscription_id
FROM bronze.billing_subscriptions s
LEFT JOIN bronze.billing_customers c ON c.customer_id = s.customer_id
WHERE c.customer_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,subscription_id


In [30]:
query = """
SELECT s.subscription_id
FROM bronze.billing_subscriptions s
LEFT JOIN bronze.billing_products p ON p.product_id = s.product_id
WHERE p.product_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,subscription_id


### Tabla invoices

In [31]:
#Perfil general
query = """
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT invoice_id) AS ids,
  COUNT(*) FILTER (WHERE issued_at IS NULL) AS issued_at,
  COUNT(*) FILTER (WHERE due_at IS NULL) AS due_at,
  COUNT(*) FILTER (WHERE total IS NULL) AS total,
  COUNT(*) FILTER (WHERE status IS NULL) AS status,
  COUNT(*) FILTER (WHERE currency IS NULL) AS currency,
  COUNT(*) FILTER (WHERE customer_id IS NULL) AS customer_id
FROM bronze.billing_invoices;
"""
df = pd.read_sql(query, engine)
df

,total,ids,issued_at,due_at,total,status,currency,customer_id
0,50000,50000,0,0,0,0,0,0


In [32]:
#Cardinalidad de status y currency
query = """
SELECT status, COUNT(*) AS n
FROM bronze.billing_invoices
GROUP BY status
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,status,n
0,paid,34966
1,pending,10048
2,overdue,4986


In [33]:
query = """
SELECT currency, COUNT(*) AS n
FROM bronze.billing_invoices
GROUP BY currency
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,currency,n
0,USD,15162
1,CLP,15099
2,EUR,5000
3,MXN,4876
4,COP,2540
5,ARS,2456
6,PEN,2455
7,BRL,2412


In [34]:
#customer_id huérfano (factura sin cliente válido)
query = """
SELECT i.invoice_id, i.customer_id
FROM bronze.billing_invoices i
LEFT JOIN bronze.billing_customers c ON c.customer_id = i.customer_id
WHERE c.customer_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,invoice_id,customer_id


In [35]:
#Validar formato de fechas antes de castear
query = """
SELECT issued_at, COUNT(*) AS n
FROM bronze.billing_invoices
WHERE issued_at IS NOT NULL AND issued_at !~ '^\d{4}-\d{2}-\d{2}'
GROUP BY issued_at;
"""
df = pd.read_sql(query, engine)
df

,issued_at,n


In [36]:
query = """
SELECT due_at, COUNT(*) AS n
FROM bronze.billing_invoices
WHERE due_at IS NOT NULL AND due_at !~ '^\d{4}-\d{2}-\d{2}'
GROUP BY due_at;
"""
df = pd.read_sql(query, engine)
df

,due_at,n


In [37]:
#due_at debe ser mayor o igual a issued_at
query = """
SELECT invoice_id, issued_at, due_at
FROM bronze.billing_invoices
WHERE due_at::date < issued_at::date;
"""
df = pd.read_sql(query, engine)
df

,invoice_id,issued_at,due_at


In [38]:
#Distribución de total (y negativos/ceros sospechosos)
query = """
SELECT
  MIN(total::numeric) AS total_min,
  MAX(total::numeric) AS total_max,
  AVG(total::numeric) AS total_promedio
FROM bronze.billing_invoices;
"""
df = pd.read_sql(query, engine)
df

,total_min,total_max,total_promedio
0,5.0,4348.24,135.837003


In [39]:
query = """
SELECT *
FROM bronze.billing_invoices
WHERE total::numeric <= 0;
"""
df = pd.read_sql(query, engine)
df

,invoice_id,issued_at,due_at,total,status,currency,customer_id


In [40]:
#total vs. suma de invoice_items (validación cruzada clave)
query = """
SELECT i.invoice_id, i.total::numeric AS total_factura,
       COALESCE(SUM(it.line_total::numeric), 0) AS suma_items,
       i.total::numeric - COALESCE(SUM(it.line_total::numeric), 0) AS diferencia
FROM bronze.billing_invoices i
LEFT JOIN bronze.billing_invoice_items it ON it.invoice_id = i.invoice_id
GROUP BY i.invoice_id, i.total
HAVING ABS(i.total::numeric - COALESCE(SUM(it.line_total::numeric), 0)) > 0.01;
"""
df = pd.read_sql(query, engine)
df

,invoice_id,total_factura,suma_items,diferencia
0,INV-00009523,49.78,1250.74,-1200.96
1,INV-00043891,121.83,1510.11,-1388.28
2,INV-00040825,124.03,1160.44,-1036.41
3,INV-00020900,82.40,478.67,-396.27
4,INV-00049247,157.82,200.96,-43.14
...,...,...,...,...
49994,INV-00048412,34.95,857.90,-822.95
49995,INV-00003688,75.01,258.80,-183.79
49996,INV-00034843,51.50,0.00,51.50
49997,INV-00036485,57.31,53.20,4.11


### Tabla invoice_items

In [41]:
#Perfil general
query = """
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT invoice_item_id) AS ids,
  COUNT(*) FILTER (WHERE quantity IS NULL) AS quantity,
  COUNT(*) FILTER (WHERE unit_price IS NULL) AS unit_price,
  COUNT(*) FILTER (WHERE line_total IS NULL) AS line_total,
  COUNT(*) FILTER (WHERE invoice_id IS NULL) AS invoice_id,
  COUNT(*) FILTER (WHERE product_id IS NULL) AS product_id
FROM bronze.billing_invoice_items;
"""
df = pd.read_sql(query, engine)
df

,total,ids,quantity,unit_price,line_total,invoice_id,product_id
0,150000,150000,0,0,0,0,0


In [42]:
#Distribución de quantity y unit_price
query = """
SELECT
  MIN(quantity::numeric) AS qty_min, MAX(quantity::numeric) AS qty_max, AVG(quantity::numeric) AS qty_avg,
  MIN(unit_price::numeric) AS precio_min, MAX(unit_price::numeric) AS precio_max, AVG(unit_price::numeric) AS precio_avg
FROM bronze.billing_invoice_items;
"""
df = pd.read_sql(query, engine)
df

,qty_min,qty_max,qty_avg,precio_min,precio_max,precio_avg
0,1.0,10.0,5.5019,1.61,697.88,42.304847


In [43]:
#Validar line_total = quantity * unit_price
query = """
SELECT invoice_item_id, quantity, unit_price, line_total
FROM bronze.billing_invoice_items
WHERE ABS(line_total::numeric - (quantity::numeric * unit_price::numeric)) > 0.01;
"""
df = pd.read_sql(query, engine)
df

,invoice_item_id,quantity,unit_price,line_total


In [44]:
#Integridad referencial
query = """
SELECT it.invoice_item_id
FROM bronze.billing_invoice_items it
LEFT JOIN bronze.billing_invoices i ON i.invoice_id = it.invoice_id
WHERE i.invoice_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,invoice_item_id


In [45]:
query = """
SELECT it.invoice_item_id
FROM bronze.billing_invoice_items it
LEFT JOIN bronze.billing_products p ON p.product_id = it.product_id
WHERE p.product_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,invoice_item_id


In [46]:
#Duplicados (mismo invoice_id + product_id)
query = """
SELECT invoice_id, product_id, COUNT(*) AS n
FROM bronze.billing_invoice_items
GROUP BY invoice_id, product_id
HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,invoice_id,product_id,n
0,INV-00023972,PRD-00067,2
1,INV-00009676,PRD-00083,2
2,INV-00045469,PRD-00190,2
3,INV-00049996,PRD-00165,2
4,INV-00014539,PRD-00024,2
...,...,...,...
1098,INV-00019759,PRD-00158,2
1099,INV-00026103,PRD-00054,2
1100,INV-00032714,PRD-00003,2
1101,INV-00015690,PRD-00002,2


### Tabla payments

In [57]:
#Perfil general
query = """
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT payment_id) AS ids,
  COUNT(*) FILTER (WHERE amount IS NULL) AS amount,
  COUNT(*) FILTER (WHERE paid_at IS NULL) AS paid_at,
  COUNT(*) FILTER (WHERE method IS NULL) AS method,
  COUNT(*) FILTER (WHERE invoice_id IS NULL) AS invoice_id
FROM bronze.billing_payments;
"""
df = pd.read_sql(query, engine)
df

,total,ids,amount,paid_at,method,invoice_id
0,80000,80000,0,0,0,0


In [48]:
#Cardinalidad de method
query = """
SELECT method, COUNT(*) AS n
FROM bronze.billing_payments
GROUP BY method
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,method,n
0,card,43990
1,bank_transfer,24054
2,paypal,7987
3,cash,3969


In [49]:
#Validar formato de paid_at antes de castear
query = """
SELECT paid_at, COUNT(*) AS n
FROM bronze.billing_payments
WHERE paid_at IS NOT NULL AND paid_at !~ '^\d{4}-\d{2}-\d{2}'
GROUP BY paid_at;
"""
df = pd.read_sql(query, engine)
df

,paid_at,n


In [50]:
#Rango de paid_at
query = """
SELECT MIN(paid_at::date) AS pago_mas_antiguo, MAX(paid_at::date) AS pago_mas_reciente
FROM bronze.billing_payments;
"""
df = pd.read_sql(query, engine)
df

,pago_mas_antiguo,pago_mas_reciente
0,2022-01-03,2026-02-13


In [51]:
#Cardinalidad de pagos por invoice (¿pagos parciales?)
query = """
SELECT invoice_id, COUNT(*) AS n_pagos
FROM bronze.billing_payments
GROUP BY invoice_id
ORDER BY n_pagos DESC;
"""
df = pd.read_sql(query, engine)
df

,invoice_id,n_pagos
0,INV-00048735,11
1,INV-00013608,11
2,INV-00008705,10
3,INV-00025853,10
4,INV-00024774,10
...,...,...
31428,INV-00018628,1
31429,INV-00040959,1
31430,INV-00024958,1
31431,INV-00037940,1


In [52]:
#Suma de pagos vs. total de la factura
query = """
SELECT i.invoice_id, i.total::numeric AS total_factura,
       COALESCE(SUM(p.amount::numeric), 0) AS total_pagado,
       i.total::numeric - COALESCE(SUM(p.amount::numeric), 0) AS pendiente
FROM bronze.billing_invoices i
LEFT JOIN bronze.billing_payments p ON p.invoice_id = i.invoice_id
GROUP BY i.invoice_id, i.total
ORDER BY pendiente DESC;
"""
df = pd.read_sql(query, engine)
df

,invoice_id,total_factura,total_pagado,pendiente
0,INV-00042681,4348.24,0.00,4348.24
1,INV-00047792,2939.01,0.00,2939.01
2,INV-00029362,2458.75,0.00,2458.75
3,INV-00009108,3157.51,727.83,2429.68
4,INV-00026619,2253.06,0.00,2253.06
...,...,...,...,...
49995,INV-00007457,2364.22,5998.61,-3634.39
49996,INV-00023222,1333.66,5693.66,-4360.00
49997,INV-00028083,1235.34,5683.90,-4448.56
49998,INV-00003278,1965.74,6645.04,-4679.30


In [53]:
#Pago con paid_at anterior al issued_at de su factura (imposible cronológicamente)
query = """
SELECT p.payment_id, p.paid_at, i.invoice_id, i.issued_at
FROM bronze.billing_payments p
JOIN bronze.billing_invoices i ON i.invoice_id = p.invoice_id
WHERE p.paid_at::date < i.issued_at::date;
"""
df = pd.read_sql(query, engine)
df

,payment_id,paid_at,invoice_id,issued_at


In [54]:
#Integridad referencial
query = """
SELECT p.payment_id
FROM bronze.billing_payments p
LEFT JOIN bronze.billing_invoices i ON i.invoice_id = p.invoice_id
WHERE i.invoice_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,payment_id
